# Drills Práticos — Cientista de Dados Itaú (Anaconda / Python)

A prova tem **parte prática** em Anaconda (sklearn). Ler não treina a mão — **isto** treina.
Cada drill segue o ciclo: **tarefa → você tenta → confere a referência**. Tudo roda **offline**, com datasets embutidos no sklearn.

**Como usar:** rode primeiro a célula de *Setup*. Em cada drill, tente na célula "Sua vez" e só depois rode a "Referência". Experimente mudar parâmetros e ver o efeito — é assim que vira músculo.

**Cobertura:** preparação de dados · regressão (R²/RMSE) · logística com **KS/Gini/AUC** · árvores/RF/boosting/AdaBoost · KNN/SVM · Naive Bayes · validação (K-Fold, holdout) · tuning (GridSearchCV) · desbalanceamento · PCA · clustering (cotovelo/silhueta).

## Setup — rode esta célula primeiro

Imports e uma função de **KS/Gini** (o sklearn não traz KS pronto, e o Itaú adora).

In [ ]:
import numpy as np, pandas as pd
import warnings; warnings.filterwarnings("ignore")
from sklearn.datasets import load_diabetes, load_breast_cancer, make_classification, make_blobs
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (roc_auc_score, confusion_matrix, classification_report,
                             mean_squared_error, mean_absolute_error, r2_score, silhouette_score)

def ks_gini(y_true, y_proba):
    """Retorna (AUC, Gini, KS) — as três réguas de crédito do Itaú."""
    auc = roc_auc_score(y_true, y_proba)
    gini = 2*auc - 1
    df = pd.DataFrame({"y": np.asarray(y_true), "p": np.asarray(y_proba)}).sort_values("p")
    cum_bom = (df.y == 0).cumsum() / max((df.y == 0).sum(), 1)
    cum_mau = (df.y == 1).cumsum() / max((df.y == 1).sum(), 1)
    ks = (cum_bom - cum_mau).abs().max()
    return auc, gini, ks

print("Setup OK — ambiente pronto.")

## Drill 1 — Carregar e explorar os dados

**Tarefa:** Carregue o dataset `load_breast_cancer` num DataFrame, e veja: dimensões, as 5 primeiras linhas e o **balanceamento** do alvo (`y`).

> Escreva sozinho na célula **Sua vez** primeiro. Só depois rode a **Referência** para conferir.

In [ ]:
# 👉 Sua vez — escreva aqui:


<details><summary>✅ ver referência</summary>

*(rode a célula abaixo)*

</details>

In [ ]:
# ✅ Referência
data = load_breast_cancer(as_frame=True)
df = data.frame
print("dimensões:", df.shape)
print(df.head())
print("\nbalanceamento do alvo:")
print(df["target"].value_counts(normalize=True).round(3))

## Drill 2 — Preparação: split, scaling e Pipeline (sem vazamento)

**Tarefa:** Separe X/y, faça `train_test_split` (estratificado, 30% teste), e monte um `Pipeline` com `StandardScaler` + um classificador. Por que Pipeline? Para o scaler aprender **só no treino**.

> Escreva sozinho na célula **Sua vez** primeiro. Só depois rode a **Referência** para conferir.

In [ ]:
# 👉 Sua vez — escreva aqui:


<details><summary>✅ ver referência</summary>

*(rode a célula abaixo)*

</details>

In [ ]:
# ✅ Referência
data = load_breast_cancer()
X, y = data.data, data.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

from sklearn.linear_model import LogisticRegression
pipe = Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(max_iter=1000))])
pipe.fit(X_tr, y_tr)                 # o scaler dá fit SÓ no treino
print("acurácia teste:", round(pipe.score(X_te, y_te), 3))
print("→ o Pipeline garante fit no treino e transform no teste (sem vazamento)")

**Bônus do Drill 2 — missing + encoding.** Numa base com valores faltantes e uma coluna categórica, impute a mediana e faça one-hot.

In [ ]:
# ✅ Referência (missing + encoding)
dfx = pd.DataFrame({"renda":[3000,5200,np.nan,8000,4100], "setor":["varejo","servico","varejo",None,"industria"]})
from sklearn.impute import SimpleImputer
dfx["renda"] = SimpleImputer(strategy="median").fit_transform(dfx[["renda"]])
dfx = pd.get_dummies(dfx, columns=["setor"], dummy_na=True)   # one-hot
print(dfx)

## Drill 3 — Regressão linear (prever um número)

**Tarefa:** No `load_diabetes`, treine uma `LinearRegression` e avalie no teste com **R²**, **RMSE** e **MAE**. Depois, olhe os **resíduos** (a média deve ser ~0).

> Escreva sozinho na célula **Sua vez** primeiro. Só depois rode a **Referência** para conferir.

In [ ]:
# 👉 Sua vez — escreva aqui:


<details><summary>✅ ver referência</summary>

*(rode a célula abaixo)*

</details>

In [ ]:
# ✅ Referência
from sklearn.linear_model import LinearRegression
dia = load_diabetes()
Xtr, Xte, ytr, yte = train_test_split(dia.data, dia.target, test_size=0.3, random_state=0)
reg = LinearRegression().fit(Xtr, ytr)
pred = reg.predict(Xte)
rmse = mean_squared_error(yte, pred) ** 0.5
print("R²  :", round(r2_score(yte, pred), 3))
print("RMSE:", round(rmse, 1))
print("MAE :", round(mean_absolute_error(yte, pred), 1))
resid = yte - pred
print("média dos resíduos (deve ~0):", round(resid.mean(), 4))

## Drill 4 — Regressão logística + KS, Gini, AUC (formato Itaú)

**Tarefa:** Treine uma logística no cancer, pegue a **probabilidade** da classe positiva (`predict_proba[:,1]`), e calcule **AUC, Gini e KS** com a função `ks_gini`. Mostre também a matriz de confusão.

> Escreva sozinho na célula **Sua vez** primeiro. Só depois rode a **Referência** para conferir.

In [ ]:
# 👉 Sua vez — escreva aqui:


<details><summary>✅ ver referência</summary>

*(rode a célula abaixo)*

</details>

In [ ]:
# ✅ Referência
from sklearn.linear_model import LogisticRegression
data = load_breast_cancer()
Xtr, Xte, ytr, yte = train_test_split(data.data, data.target, test_size=0.3, stratify=data.target, random_state=42)
clf = Pipeline([("s", StandardScaler()), ("lr", LogisticRegression(max_iter=1000))]).fit(Xtr, ytr)
proba = clf.predict_proba(Xte)[:, 1]            # probabilidade da classe 1
auc, gini, ks = ks_gini(yte, proba)
print(f"AUC = {auc:.3f} | Gini = {gini:.3f} | KS = {ks:.3f}")
print("\nmatriz de confusão (limiar 0.5):")
print(confusion_matrix(yte, (proba >= 0.5).astype(int)))

# experimente: mude o limiar para 0.3 e veja recall subir
pred03 = (proba >= 0.3).astype(int)
print("\ncom limiar 0.3:"); print(confusion_matrix(yte, pred03))

## Drill 5 — Árvore, Random Forest, Gradient Boosting e AdaBoost

**Tarefa:** Compare quatro modelos pelo **AUC** no cancer: `DecisionTree`, `RandomForest`, `GradientBoosting` e `AdaBoost`. Qual vence? (boosting costuma liderar em tabular.)

> Escreva sozinho na célula **Sua vez** primeiro. Só depois rode a **Referência** para conferir.

In [ ]:
# 👉 Sua vez — escreva aqui:


<details><summary>✅ ver referência</summary>

*(rode a célula abaixo)*

</details>

In [ ]:
# ✅ Referência
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
data = load_breast_cancer()
Xtr, Xte, ytr, yte = train_test_split(data.data, data.target, test_size=0.3, stratify=data.target, random_state=42)
modelos = {
    "Árvore": DecisionTreeClassifier(max_depth=4, random_state=0),
    "RandomForest": RandomForestClassifier(n_estimators=300, random_state=0),
    "GradientBoosting": GradientBoostingClassifier(random_state=0),
    "AdaBoost": AdaBoostClassifier(random_state=0),
}
for nome, m in modelos.items():
    m.fit(Xtr, ytr)
    auc = roc_auc_score(yte, m.predict_proba(Xte)[:, 1])
    print(f"{nome:18s} AUC = {auc:.3f}")

## Drill 6 — KNN e SVM (e por que padronizar importa)

**Tarefa:** Treine um `KNN` (varie `n_neighbors`) e um `SVM` (`SVC` com `probability=True`), **dentro de um Pipeline com StandardScaler**. Compare o AUC. Lembre: ambos dependem de escala.

> Escreva sozinho na célula **Sua vez** primeiro. Só depois rode a **Referência** para conferir.

In [ ]:
# 👉 Sua vez — escreva aqui:


<details><summary>✅ ver referência</summary>

*(rode a célula abaixo)*

</details>

In [ ]:
# ✅ Referência
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
data = load_breast_cancer()
Xtr, Xte, ytr, yte = train_test_split(data.data, data.target, test_size=0.3, stratify=data.target, random_state=42)

for k in [3, 5, 11]:
    knn = Pipeline([("s", StandardScaler()), ("knn", KNeighborsClassifier(n_neighbors=k))]).fit(Xtr, ytr)
    print(f"KNN (k={k:2d}) AUC = {roc_auc_score(yte, knn.predict_proba(Xte)[:,1]):.3f}")

svm = Pipeline([("s", StandardScaler()), ("svc", SVC(C=1.0, kernel="rbf", probability=True))]).fit(Xtr, ytr)
print(f"SVM (rbf, C=1)  AUC = {roc_auc_score(yte, svm.predict_proba(Xte)[:,1]):.3f}")
# experimente: tire o StandardScaler do KNN e veja o AUC cair

## Drill 7 — Naive Bayes (Gaussiano)

**Tarefa:** Treine um `GaussianNB` no cancer (features contínuas → variante Gaussiana) e avalie por AUC.

> Escreva sozinho na célula **Sua vez** primeiro. Só depois rode a **Referência** para conferir.

In [ ]:
# 👉 Sua vez — escreva aqui:


<details><summary>✅ ver referência</summary>

*(rode a célula abaixo)*

</details>

In [ ]:
# ✅ Referência
from sklearn.naive_bayes import GaussianNB
data = load_breast_cancer()
Xtr, Xte, ytr, yte = train_test_split(data.data, data.target, test_size=0.3, stratify=data.target, random_state=42)
nb = GaussianNB().fit(Xtr, ytr)
print("GaussianNB AUC =", round(roc_auc_score(yte, nb.predict_proba(Xte)[:,1]), 3))
print("(features contínuas → Gaussian; texto/contagem → Multinomial; binárias → Bernoulli)")

## Drill 8 — Validação: K-Fold e holdout

**Tarefa:** Use `cross_val_score` com `KFold(n_splits=5, shuffle=False)` (o padrão da prova!) e métrica `roc_auc`. Compare a média da CV com um único holdout. Qual estimativa é mais estável?

> Escreva sozinho na célula **Sua vez** primeiro. Só depois rode a **Referência** para conferir.

In [ ]:
# 👉 Sua vez — escreva aqui:


<details><summary>✅ ver referência</summary>

*(rode a célula abaixo)*

</details>

In [ ]:
# ✅ Referência
from sklearn.linear_model import LogisticRegression
data = load_breast_cancer(); X, y = data.data, data.target
pipe = Pipeline([("s", StandardScaler()), ("lr", LogisticRegression(max_iter=1000))])

kf = KFold(n_splits=5, shuffle=False)               # ATENÇÃO: shuffle=False (padrão Itaú)
scores = cross_val_score(pipe, X, y, cv=kf, scoring="roc_auc")
print("AUC por dobra:", np.round(scores, 3))
print(f"CV 5-fold: média = {scores.mean():.3f} (+/- {scores.std():.3f})")

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, stratify=y, random_state=1)
pipe.fit(Xtr, ytr)
print("holdout único:", round(roc_auc_score(yte, pipe.predict_proba(Xte)[:,1]), 3), "(depende da fatia)")

## Drill 9 — Tuning de hiperparâmetros (GridSearchCV + Pipeline)

**Tarefa:** Use `GridSearchCV` para achar o melhor `C` e `gamma` de um SVM, **dentro de um Pipeline** (para o scaling não vazar na CV). Mostre os `best_params_` e o melhor AUC.

> Escreva sozinho na célula **Sua vez** primeiro. Só depois rode a **Referência** para conferir.

In [ ]:
# 👉 Sua vez — escreva aqui:


<details><summary>✅ ver referência</summary>

*(rode a célula abaixo)*

</details>

In [ ]:
# ✅ Referência
from sklearn.svm import SVC
data = load_breast_cancer(); X, y = data.data, data.target
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
pipe = Pipeline([("s", StandardScaler()), ("svc", SVC(probability=True))])
grid = {"svc__C": [0.1, 1, 10], "svc__gamma": [0.001, 0.01, 0.1]}
busca = GridSearchCV(pipe, grid, cv=5, scoring="roc_auc", n_jobs=-1).fit(Xtr, ytr)
print("melhores parâmetros:", busca.best_params_)
print("melhor AUC (CV):", round(busca.best_score_, 3))
print("AUC no teste:", round(roc_auc_score(yte, busca.predict_proba(Xte)[:,1]), 3))

## Drill 10 — Dados desbalanceados

**Tarefa:** Crie uma base desbalanceada (`make_classification`, weights=[0.97,0.03]). Compare uma logística **sem** e **com** `class_weight='balanced'` pelo **recall** da classe rara. (Bônus: SMOTE, se tiver `imbalanced-learn`.)

> Escreva sozinho na célula **Sua vez** primeiro. Só depois rode a **Referência** para conferir.

In [ ]:
# 👉 Sua vez — escreva aqui:


<details><summary>✅ ver referência</summary>

*(rode a célula abaixo)*

</details>

In [ ]:
# ✅ Referência
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score
X, y = make_classification(n_samples=4000, weights=[0.97, 0.03], n_features=12, random_state=7)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, stratify=y, random_state=7)

base = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
bal  = LogisticRegression(max_iter=1000, class_weight="balanced").fit(Xtr, ytr)
print("recall classe rara — sem peso :", round(recall_score(yte, base.predict(Xte)), 3))
print("recall classe rara — balanced :", round(recall_score(yte, bal.predict(Xte)), 3))

# Bônus SMOTE (opcional):
try:
    from imblearn.over_sampling import SMOTE
    Xs, ys = SMOTE(random_state=7).fit_resample(Xtr, ytr)   # SÓ no treino
    sm = LogisticRegression(max_iter=1000).fit(Xs, ys)
    print("recall classe rara — SMOTE    :", round(recall_score(yte, sm.predict(Xte)), 3))
except ImportError:
    print("(SMOTE indisponível — instale com: pip install imbalanced-learn)")

## Drill 11 — PCA (redução de dimensionalidade)

**Tarefa:** No cancer, **padronize** e aplique PCA. Imprima a **variância explicada** por componente e a acumulada. Depois, use `PCA(n_components=0.95)` para guardar 95% da variância — quantos componentes sobram?

> Escreva sozinho na célula **Sua vez** primeiro. Só depois rode a **Referência** para conferir.

In [ ]:
# 👉 Sua vez — escreva aqui:


<details><summary>✅ ver referência</summary>

*(rode a célula abaixo)*

</details>

In [ ]:
# ✅ Referência
from sklearn.decomposition import PCA
data = load_breast_cancer()
Xs = StandardScaler().fit_transform(data.data)        # padronizar ANTES é obrigatório
pca = PCA().fit(Xs)
var = pca.explained_variance_ratio_
print("variância dos 5 primeiros PCs:", np.round(var[:5], 3))
print("acumulada (5 primeiros):", np.round(np.cumsum(var[:5]), 3))
pca95 = PCA(n_components=0.95).fit(Xs)
print(f"\npara guardar 95% da variância bastam {pca95.n_components_} componentes (de {data.data.shape[1]})")

## Drill 12 — Clustering: cotovelo e silhueta

**Tarefa:** Gere grupos com `make_blobs` (4 centros). Rode `KMeans` para k=2..7 e plote a **inércia** (cotovelo) e a **silhueta**. Qual k os dois métodos sugerem?

> Escreva sozinho na célula **Sua vez** primeiro. Só depois rode a **Referência** para conferir.

In [ ]:
# 👉 Sua vez — escreva aqui:


<details><summary>✅ ver referência</summary>

*(rode a célula abaixo)*

</details>

In [ ]:
# ✅ Referência
from sklearn.cluster import KMeans
X, _ = make_blobs(n_samples=600, centers=4, cluster_std=1.0, random_state=42)
print(f"{'k':>2} {'inércia':>10} {'silhueta':>10}")
for k in range(2, 8):
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X)
    sil = silhouette_score(X, km.labels_)
    print(f"{k:>2} {km.inertia_:>10.0f} {sil:>10.3f}")
print("\n↑ a inércia 'dobra o cotovelo' e a silhueta é máxima no k verdadeiro (=4)")

## Pronto — agora é repetição

Refaça os drills **sem olhar a referência** até sair natural. Para virar prova de verdade:
1. Tente reescrever cada drill do zero, só com a tarefa.
2. Misture: pegue um drill de classificação e troque o modelo, recalculando AUC/KS.
3. Cronometre — a parte prática também tem tempo.

Combine com o **Simulado Cronometrado** (a parte teórica) e você fecha as duas frentes da prova.